# 🛰️ EuroSAT Land-Use Classification — Custom CNN vs. ResNet50

**MSc AIVC — University of West Attica · Ergasia 2026**

This notebook trains and compares two models on the EuroSAT dataset:
1. a **Custom CNN** trained *from scratch*, and
2. a **ResNet50** adapted via *transfer learning + fine-tuning*.

It downloads the data, trains both models, runs an **Optuna (5-trial)** hyperparameter
search, and produces loss/accuracy curves, confusion matrices and prediction grids.

> **No Kaggle token / account needed** — the dataset downloads automatically from the
> public Zenodo mirror.
>
> **Tip:** on Colab choose a **GPU** runtime (*Runtime → Change runtime type → GPU*).

## 1. Setup

In [ ]:
# Install dependencies (Colab). Locally, use: pip install -r requirements.txt
import sys
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    !pip -q install optuna seaborn

In [ ]:
# Make the `src` package importable both on Colab (after git clone) and locally.
import os, sys
from pathlib import Path

if IN_COLAB and not Path("src").exists() and not Path("deep").exists():
    # Clone your repository, then cd into it:
    # !git clone https://github.com/Emma-nouela/eurosat-cnn-resnet.git deep
    # %cd deep
    print("On Colab: clone your repo (see the commented commands above), then re-run this cell.")

ROOT = Path.cwd()
if (ROOT / "src").exists():
    sys.path.insert(0, str(ROOT))
elif (ROOT.parent / "src").exists():     # running from notebooks/
    sys.path.insert(0, str(ROOT.parent))
print("Project root:", ROOT)

In [ ]:
import torch
from src import config, utils
from src.data import get_dataloaders
from src.models import build_model, count_parameters
from src.train import train_model
from src.evaluate import evaluate
from src.tune import run_study

utils.set_seed()
print("Device:", config.DEVICE)
print("Classes:", config.CLASSES)

## 2. Dataset exploration

The dataset downloads automatically from Zenodo on first use (no token). If you would
rather use the Kaggle source instead, call `download_dataset(source="kaggle")` after
configuring `kaggle.json`.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np, collections
from PIL import Image
from torchvision import datasets
from src.data import download_dataset

img_root = download_dataset()          # token-free Zenodo download
preview = datasets.ImageFolder(str(img_root))
print("Total images:", len(preview))

# Class distribution
counts = collections.Counter([preview.classes[t] for t in preview.targets])
plt.figure(figsize=(10, 4))
plt.bar(counts.keys(), counts.values())
plt.xticks(rotation=45, ha="right"); plt.title("EuroSAT class distribution"); plt.tight_layout(); plt.show()

In [ ]:
# One sample image per class
fig, axes = plt.subplots(2, 5, figsize=(15, 6))
seen = {}
for path, label in preview.samples:
    name = preview.classes[label]
    seen.setdefault(name, path)
    if len(seen) == len(preview.classes):
        break
for ax, (name, path) in zip(axes.ravel(), seen.items()):
    ax.imshow(Image.open(path)); ax.set_title(name, fontsize=10); ax.axis("off")
plt.suptitle("EuroSAT — one sample per class"); plt.tight_layout(); plt.show()

## 3. The Idea (methodology)

We compare two of the assignment's approaches under an identical data pipeline
(reproducible stratified 70/15/15 split, augmentation, Adam + `ReduceLROnPlateau`):

* **Custom CNN** — 4 Conv-BN-ReLU-MaxPool blocks → global average pool → MLP head.
  Trained from scratch on native 64×64 tiles. Lightweight baseline.
* **ResNet50** — ImageNet-pretrained backbone with a new 10-way head, fully fine-tuned
  on 224×224 inputs. Transfer learning for higher accuracy and faster convergence.

## 4. Train the Custom CNN

In [ ]:
cnn_train, cnn_val, cnn_test, classes = get_dataloaders("cnn")
cnn = build_model("cnn")
print("Trainable params (CNN):", f"{count_parameters(cnn):,}")

cnn, cnn_hist, cnn_best = train_model(
    cnn, cnn_train, cnn_val,
    epochs=config.DEFAULTS["cnn"]["epochs"],
    lr=config.DEFAULTS["cnn"]["lr"],
    ckpt_path=config.RESULTS_DIR / "cnn_best.pth",
)
utils.plot_curves(cnn_hist, "Custom CNN", save_path=config.RESULTS_DIR / "cnn_curves.png")
plt.show()
print("Best val accuracy (CNN):", round(cnn_best, 4))

## 5. Train ResNet50 (transfer learning)

In [ ]:
rn_train, rn_val, rn_test, _ = get_dataloaders("resnet")
resnet = build_model("resnet", pretrained=True)
print("Trainable params (ResNet50):", f"{count_parameters(resnet):,}")

resnet, rn_hist, rn_best = train_model(
    resnet, rn_train, rn_val,
    epochs=config.DEFAULTS["resnet"]["epochs"],
    lr=config.DEFAULTS["resnet"]["lr"],
    ckpt_path=config.RESULTS_DIR / "resnet_best.pth",
)
utils.plot_curves(rn_hist, "ResNet50", save_path=config.RESULTS_DIR / "resnet_curves.png")
plt.show()
print("Best val accuracy (ResNet50):", round(rn_best, 4))

## 6. Hyperparameter optimization — Optuna (5 trials)

We tune the **Custom CNN** (cheapest to search) over learning rate, dropout, batch size
and optimizer. Switch to `model_type="resnet"` to tune ResNet instead.

In [ ]:
study = run_study(model_type="cnn", n_trials=5, tune_epochs=5)
print("Best params:", study.best_params, "| best val acc:", round(study.best_value, 4))

# Retrain the final CNN with the tuned hyperparameters:
best = study.best_params
tuned_train, tuned_val, _, _ = get_dataloaders("cnn", batch_size=best["batch_size"])
tuned_cnn = build_model("cnn", dropout=best["dropout"])
tuned_cnn, tuned_hist, tuned_best = train_model(
    tuned_cnn, tuned_train, tuned_val,
    epochs=config.DEFAULTS["cnn"]["epochs"],
    lr=best["lr"], optimizer_name=best["optimizer"],
    ckpt_path=config.RESULTS_DIR / "cnn_best.pth",   # overwrite with the tuned model
)
utils.plot_curves(tuned_hist, "Custom CNN (tuned)", save_path=config.RESULTS_DIR / "cnn_curves.png")
plt.show()
print("Tuned CNN best val accuracy:", round(tuned_best, 4))

## 7. Analysis — evaluation on the test set

In [ ]:
cnn_acc, cnn_report = evaluate("cnn")
plt.show()

In [ ]:
rn_acc, rn_report = evaluate("resnet")
plt.show()

## 8. Comparison

In [ ]:
import pandas as pd

def macro_f1(rep):
    return rep["macro avg"]["f1-score"]

table = pd.DataFrame([
    {"Model": "Custom CNN", "Approach": "from scratch",
     "Test accuracy": round(cnn_acc, 4), "Macro F1": round(macro_f1(cnn_report), 4),
     "Trainable params": count_parameters(build_model("cnn"))},
    {"Model": "ResNet50", "Approach": "transfer learning",
     "Test accuracy": round(rn_acc, 4), "Macro F1": round(macro_f1(rn_report), 4),
     "Trainable params": count_parameters(build_model("resnet"))},
])
table.to_csv(config.RESULTS_DIR / "comparison.csv", index=False)
table

### Discussion

* **ResNet50 (transfer learning)** typically reaches the highest accuracy and converges
  in fewer epochs — ImageNet features transfer well to satellite imagery — at the cost of
  ~20× more parameters and heavier compute.
* **The Custom CNN** is a lightweight, fast baseline that still performs strongly on the
  native 64×64 tiles, but has a lower accuracy ceiling.
* **Common confusions** (see the confusion matrices): `Highway`↔`River`,
  `PermanentCrop`↔`AnnualCrop`, `HerbaceousVegetation`↔`Pasture`.

All figures are saved in `results/` and embedded in the project `README.md`.